# Project 8: Capstone — End-to-End ML Pipeline

**Chapters:** All (1–10)

**Goal:** Tie everything together. Take a real-ish dataset through the full ML pipeline: explore, preprocess, try multiple models (from perceptron to neural net), compare them fairly, and report what you learned.

**What you'll do:**
1. Load and explore the MNIST digits subset (8×8, from sklearn)
2. Apply models you built in Projects 1–7
3. Compare: perceptron, SVM, kernel methods, neural net, CNN
4. Write a mini-report with your findings

**Rules:** Use your own implementations from earlier projects. You may use sklearn *only* for data loading and train/test splitting.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

np.random.seed(42)
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
print("All ran")

## 1. Load & Explore the Data

We'll use sklearn's `load_digits` — 8×8 grayscale images of handwritten digits. For simplicity, we'll start with a binary classification: **digit 3 vs digit 8** (they look similar!).

In [ ]:
digits = load_digits()
print(f"Full dataset: {digits.data.shape[0]} images, {digits.data.shape[1]} features (8x8 pixels)")
print(f"Labels: {np.unique(digits.target)}")

# Binary subset: 3 vs 8
mask = (digits.target == 3) | (digits.target == 8)
X_all = digits.data[mask]
y_all = digits.target[mask]
y_binary = np.where(y_all == 3, -1, 1)  # 3 → -1, 8 → +1

# Normalize to [0, 1]
X_all = X_all / 16.0

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_binary, test_size=0.3, random_state=42, stratify=y_binary
)

print(f"\nBinary (3 vs 8): {len(X_all)} images")
print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Class balance: {np.mean(y_train==-1):.1%} threes, {np.mean(y_train==1):.1%} eights")

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(18, 4))
for row, label in enumerate([-1, 1]):
    subset = X_train[y_train == label]
    for col in range(10):
        axes[row, col].imshow(subset[col].reshape(8, 8), cmap='gray')
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel('3' if label == -1 else '8', fontsize=16, rotation=0, labelpad=20)
plt.suptitle('Sample Digits: 3 (top) vs 8 (bottom)', fontsize=14)
plt.tight_layout(); plt.show()

## 1b. Evaluation Framework

Before training models, set up a standard evaluation pipeline. Every model will be measured the same way.

In [ ]:
def evaluate_model(model_name, y_true, y_pred):
    """Compute standard metrics for a binary classifier.
    
    Args:
        model_name: string label for this model
        y_true: ground truth labels (+1/-1)
        y_pred: predicted labels (+1/-1)
    
    Returns:
        dict with model name, accuracy, precision, recall, F1
    """
    acc = np.mean(y_true == y_pred)
    
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == -1))
    fn = np.sum((y_pred == -1) & (y_true == 1))
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {
        'Model': model_name,
        'Accuracy': acc,
        'Precision': precision,
        'Recall': recall,
        'F1': f1
    }

def confusion_matrix(y_true, y_pred):
    """2x2 confusion matrix for +1/-1 labels.
    Returns [[TN, FP], [FN, TP]]."""
    tp = np.sum((y_pred == 1) & (y_true == 1))
    tn = np.sum((y_pred == -1) & (y_true == -1))
    fp = np.sum((y_pred == 1) & (y_true == -1))
    fn = np.sum((y_pred == -1) & (y_true == 1))
    return np.array([[tn, fp], [fn, tp]])

# Collect results here — append after each model
all_results = []
print("Evaluation framework ready. After each model, call:")
print("  all_results.append(evaluate_model('ModelName', y_test, predictions))")

## 2. Apply Your Models

Copy your implementations from Projects 1–7 (or import them). We'll test each one.

### Model A: Perceptron (Project 1)

In [ ]:
# Copy or adapt your perceptron from Project 1 here
# Key change: input is now 64-dimensional, not 2D

### YOUR CODE ###
# 1. Train on X_train, y_train (labels are +1/-1)
# 2. Evaluate on X_test, y_test
# 3. Store the accuracy

perceptron_acc = None  # Replace with actual accuracy
print(f"Perceptron test accuracy: {perceptron_acc}")

### Model B: Linear SVM (Project 2)

In [ ]:
# Copy or adapt your LinearSVM from Project 2
# Try different λ values

### YOUR CODE ###

svm_acc = None
print(f"SVM test accuracy: {svm_acc}")

### Model C: Kernel Perceptron with RBF (Project 4)

In [ ]:
# Copy or adapt your KernelPerceptron + rbf_kernel from Project 4
# Note: RBF on 64D data may be slow — start with a small subset to debug

### YOUR CODE ###

kernel_acc = None
print(f"Kernel Perceptron test accuracy: {kernel_acc}")

### Model D: Neural Network (Project 5)

Adapt your TwoLayerNet for 64 inputs and binary output.

In [ ]:
# Copy or adapt your TwoLayerNet from Project 5
# Key change: n_input=64, need to convert labels from {-1,+1} to {0,1}

### YOUR CODE ###

nn_acc = None
print(f"Neural Network test accuracy: {nn_acc}")

## 3. Compare Results

In [ ]:
if not all_results:
    print("No results yet — implement the models above and append to all_results.")
else:
    # --- Formatted results table ---
    header = f"{'Model':<30} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>6}"
    print(header)
    print("-" * len(header))
    for r in all_results:
        print(f"{r['Model']:<30} {r['Accuracy']:>8.1%} {r['Precision']:>9.1%} {r['Recall']:>7.1%} {r['F1']:>6.3f}")
    
    # --- Bar chart of all metrics ---
    models = [r['Model'] for r in all_results]
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
    x = np.arange(len(models))
    width = 0.2
    colors = ['#4a90d9', '#5cb85c', '#f0ad4e', '#d9534f']
    
    fig, ax = plt.subplots(figsize=(12, 5))
    for i, (metric, color) in enumerate(zip(metrics, colors)):
        vals = [r[metric] for r in all_results]
        ax.bar(x + i * width, vals, width, label=metric, color=color)
    ax.set_xticks(x + 1.5 * width)
    ax.set_xticklabels(models, rotation=15, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title('Model Comparison: All Metrics')
    ax.legend()
    plt.tight_layout(); plt.show()
    
    # --- Confusion matrices side by side ---
    n_models = len(all_results)
    fig, axes = plt.subplots(1, n_models, figsize=(4 * n_models, 3.5))
    if n_models == 1:
        axes = [axes]
    
    # You need to store predictions for each model to plot confusion matrices.
    # Example: all_preds = {'Perceptron': preds_perceptron, 'SVM': preds_svm, ...}
    # If you haven't stored predictions, this section shows a placeholder.
    print("\nTo see confusion matrices, store predictions in a dict called all_preds:")
    print("  all_preds = {'Perceptron': y_pred_perceptron, 'SVM': y_pred_svm, ...}")
    
    if 'all_preds' in dir():
        for ax, r in zip(axes, all_results):
            name = r['Model']
            if name in all_preds:
                cm = confusion_matrix(y_test, all_preds[name])
                ax.imshow(cm, cmap='Blues', vmin=0)
                for i in range(2):
                    for j in range(2):
                        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=14,
                                color='white' if cm[i, j] > cm.max() / 2 else 'black')
                ax.set_xticks([0, 1]); ax.set_xticklabels(['Pred 3', 'Pred 8'])
                ax.set_yticks([0, 1]); ax.set_yticklabels(['True 3', 'True 8'])
                ax.set_title(name, fontsize=11)
        plt.suptitle('Confusion Matrices', fontsize=13)
        plt.tight_layout(); plt.show()

## 4. Multi-Class Extension: All 10 Digits

Binary classification was the warmup. Now tackle the real thing — classify **all 10 digits** (0–9) using **One-vs-All (OvA)**.

**How OvA works:**
1. Train 10 binary classifiers: digit-k vs everything-else, for k = 0, 1, ..., 9
2. For each test sample, run all 10 classifiers
3. Predict the class whose classifier is most confident (highest score)

**Your task:**
- Use your best-performing binary model from above (e.g., SVM or neural net)
- Train 10 OvA classifiers on the full digit dataset
- Compute the 10×10 confusion matrix — which digit pairs get confused most?

In [ ]:
# Full 10-digit dataset
X_multi = digits.data / 16.0
y_multi = digits.target

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y_multi, test_size=0.3, random_state=42, stratify=y_multi
)
print(f"Multi-class: {len(X_train_m)} train, {len(X_test_m)} test, {len(np.unique(y_multi))} classes")

### YOUR CODE ###
# 1. For each digit k (0-9):
#    - Create binary labels: +1 where y_train_m == k, -1 otherwise
#    - Train your chosen binary classifier
#    - Store the classifier (or its decision scores on the test set)
#
# 2. For each test sample, predict the digit with the highest score
#
# 3. Compute accuracy and the 10x10 confusion matrix

raise NotImplementedError("One-vs-All classification")

# --- After implementing, this visualizes the 10x10 confusion matrix ---
# Uncomment once you have y_pred_multi:
#
# cm_multi = np.zeros((10, 10), dtype=int)
# for true, pred in zip(y_test_m, y_pred_multi):
#     cm_multi[true, pred] += 1
#
# fig, ax = plt.subplots(figsize=(9, 8))
# ax.imshow(cm_multi, cmap='Blues')
# for i in range(10):
#     for j in range(10):
#         ax.text(j, i, str(cm_multi[i, j]), ha='center', va='center', fontsize=9,
#                 color='white' if cm_multi[i, j] > cm_multi.max() / 2 else 'black')
# ax.set_xticks(range(10)); ax.set_yticks(range(10))
# ax.set_xlabel('Predicted'); ax.set_ylabel('True')
# ax.set_title(f'10-Class Confusion Matrix (Accuracy: {np.mean(y_pred_multi == y_test_m):.1%})')
# plt.tight_layout(); plt.show()
#
# # Which pairs are most confused?
# np.fill_diagonal(cm_multi, 0)
# worst = np.unravel_index(cm_multi.argmax(), cm_multi.shape)
# print(f"Most confused pair: digit {worst[0]} ↔ digit {worst[1]} ({cm_multi[worst[0], worst[1]]} errors)")

## 5. Mini-Report

Write 3-5 paragraphs covering:

1. **Which model won?** Why do you think it performed best on this data?
2. **Surprises:** Did any model do better/worse than expected? Why?
3. **Linear vs nonlinear:** Was a nonlinear model necessary, or did linear models do well enough?
4. **If you had more time:** What would you try next? More data? Different features? Deeper networks?
5. **Connection to the course:** Which course concept turned out to be most important for this task?

---

*Your report:*

...